# חיים קל: ייצור האודיו לאימון על GPU (Colab)

המודל הכבד (Chatterbox Multilingual, PyTorch, MIT) מקריא כאן את המשפטים בקול של חיים. לפני ההרצה להעלות ל-Google Drive, לתיקייה `otzaria-light/gen-input/`:

* `sentences.tsv` ו-`sentences.vocalized.txt` (מ-`select_text.py` ו-`--vocalize --marks` במחשב)
* `ref.wav`: 10–20 שניות נקיות של הקול (הגייה רגילה)

הפלט: קובצי zip של 200 קליפים ב-`otzaria-light/gen-output/`, שממשיכים מאיפה שנעצרו. מורידים אותם למחשב ומריצים `unpack_colab_output.py`, ואז `qc.py` כרגיל.

Runtime ← Change runtime type ← GPU (L4 או A100 מהירים פי 2–3 מ-T4).

In [ ]:
# 1. Drive, folders, settings
from google.colab import drive
drive.mount('/content/drive')
import os, glob
ROOT = '/content/drive/MyDrive/otzaria-light'
IN, OUT = f'{ROOT}/gen-input', f'{ROOT}/gen-output'
os.makedirs(OUT, exist_ok=True)
START, END = 0, 0          # lines of sentences.tsv to do (END=0: all); e.g. 1500,0 skips the laptop's pilot
ZIP_EVERY = 200            # clips per zip written to Drive
for f in ('sentences.tsv', 'sentences.vocalized.txt', 'ref.wav'):
    assert os.path.exists(f'{IN}/{f}'), f'missing {IN}/{f}'
!nvidia-smi -L

In [ ]:
# 2. Chatterbox (the model files, ~2 GB, are fetched from Hugging Face on first load)
!pip install -q chatterbox-tts soundfile
import torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
model = ChatterboxMultilingualTTS.from_pretrained(device='cuda')
model.prepare_conditionals(f'{IN}/ref.wav', exaggeration=0.5)
print('sample rate', model.sr)

In [ ]:
# 3. The sentences (text for the model without the pronunciation marks), minus what is done already
import re, json, zipfile
MARKS = re.compile('[\u05AB\u05BD|]')
rows = []
with open(f'{IN}/sentences.tsv', encoding='utf-8') as tsv, open(f'{IN}/sentences.vocalized.txt', encoding='utf-8') as voc:
    next(tsv)
    for meta, text in zip(tsv, voc):
        sid, source, book, vocalized, _ = meta.rstrip('\n').split('\t', 4)
        rows.append((sid, source, book, text.rstrip('\n')))
rows = rows[START:END or None]
done = set()
for z in glob.glob(f'{OUT}/clips_*.zip'):
    with zipfile.ZipFile(z) as zf:
        done.update(n[:-4] for n in zf.namelist() if n.endswith('.wav'))
todo = [r for r in rows if r[0] not in done]
print(f'{len(rows)} lines, {len(done)} done, {len(todo)} to do')

In [ ]:
# 4. Generation: pieces of up to 80 letters (as the service does), a short pause between them,
#    silence trimmed, peak 0.89, 22.05 kHz 16-bit; every ZIP_EVERY clips go to Drive as one zip
import io, time, numpy as np, soundfile as sf
NIKUD = re.compile('[\u0591-\u05C7]')

def pieces(text, target=80):
    out, cur, letters, last_comma, last_space = [], '', 0, -1, -1
    def flush(upto):
        nonlocal cur, letters, last_comma, last_space
        head, cur = cur[:upto].strip(), cur[upto:]
        if head: out.append(head)
        letters = len(NIKUD.sub('', cur)); last_comma = last_space = -1
    for ch in text:
        cur += ' ' if ch in '\n\r\t' else ch
        if not NIKUD.match(ch): letters += 1
        if ch in '.?!:;\u05C3\n': flush(len(cur)); continue
        if ch == ',': last_comma = len(cur)
        if ch == ' ': last_space = len(cur)
        if letters >= target:
            if last_comma > len(cur) / 3: flush(last_comma)
            elif last_space > 0: flush(last_space)
            else: flush(len(cur))
    flush(len(cur))
    return out

def trim(x, rate, db=-40.0, pad=0.06):
    frame = int(rate * 0.01); n = len(x) // frame
    if n < 3: return x
    rms = np.sqrt((x[:n*frame].reshape(n, frame) ** 2).mean(axis=1) + 1e-12)
    loud = np.where(20 * np.log10(rms) > db)[0]
    if len(loud) == 0: return x
    a = max(0, loud[0] * frame - int(rate * pad)); b = min(len(x), (loud[-1] + 1) * frame + int(rate * pad))
    return x[a:b]

resample = torchaudio.transforms.Resample(model.sr, 22050).to('cuda')
batch, meta, zip_index = {}, [], len(glob.glob(f'{OUT}/clips_*.zip'))
t0 = time.time(); seconds_total = 0.0
def flush_zip():
    global batch, meta, zip_index
    if not batch: return
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_STORED) as zf:
        for sid, wav in batch.items(): zf.writestr(sid + '.wav', wav)
        zf.writestr('clips.jsonl', '\n'.join(json.dumps(m, ensure_ascii=False) for m in meta) + '\n')
    zip_index += 1
    with open(f'{OUT}/clips_{zip_index:04d}.zip', 'wb') as f: f.write(buf.getvalue())
    print(f'  wrote clips_{zip_index:04d}.zip ({len(batch)} clips); {seconds_total/3600:.2f} h of audio in {(time.time()-t0)/3600:.2f} h')
    batch, meta = {}, []

with torch.inference_mode():
    for i, (sid, source, book, text) in enumerate(todo, 1):
        parts = []
        for piece in pieces(MARKS.sub('', text)):
            wav = model.generate(piece, language_id='he', exaggeration=0.5, cfg_weight=0.5)
            parts.append(resample(wav.to('cuda')).squeeze(0).cpu().numpy())
            parts.append(np.zeros(22050 // 6, dtype=np.float32))
        x = trim(np.concatenate(parts[:-1]), 22050)
        peak = float(np.abs(x).max()) if len(x) else 0.0
        if peak > 0: x = x * (0.89 / peak)
        buf = io.BytesIO(); sf.write(buf, np.clip(x, -1, 1), 22050, format='WAV', subtype='PCM_16')
        batch[sid] = buf.getvalue(); seconds_total += len(x) / 22050
        meta.append({'id': sid, 'source': source, 'book': book, 'text': text, 'seconds': round(len(x) / 22050, 2), 'ms': 0})
        if len(batch) >= ZIP_EVERY: flush_zip()
        if i % 50 == 0: print(f'{i}/{len(todo)}  {seconds_total/60:.1f} min of audio, {(time.time()-t0)/60:.1f} min elapsed')
flush_zip()
print('done')